In [1]:
import json
import os
from urllib.parse import urlencode
from urllib.request import urlopen

from dotenv import load_dotenv

load_dotenv()

def get_current_weather(location: str, unit: str = ""):
    """
    Get the current weather in a given location.

    location (str): The city and state, e.g. Madrid, Barcelona
    unit (str): Optional. It can take two values; "celsius", "fahrenheit".
        If omitted, the function returns both Celsius and Fahrenheit.
    """
    api_key = os.getenv("WEATHER_API_KEY")
    if not api_key:
        raise ValueError("WEATHER_API_KEY is not set. Add it to your .env file.")

    normalized_unit = unit.strip().lower() if unit else ""
    if normalized_unit and normalized_unit not in {"celsius", "fahrenheit"}:
        raise ValueError("unit must be either 'celsius', 'fahrenheit', or omitted")

    query = urlencode({"key": api_key, "q": location})
    url = f"http://api.weatherapi.com/v1/current.json?{query}"

    with urlopen(url) as response:
        payload = json.loads(response.read().decode("utf-8"))

    current = payload["current"]
    result = {
        "location": payload["location"]["name"],
        "condition": current["condition"]["text"],
    }

    if normalized_unit == "celsius":
        result["temperature"] = current["temp_c"]
        result["unit"] = "celsius"
    elif normalized_unit == "fahrenheit":
        result["temperature"] = current["temp_f"]
        result["unit"] = "fahrenheit"
    else:
        result["temperature_c"] = current["temp_c"]
        result["temperature_f"] = current["temp_f"]

    return json.dumps(result)

In [2]:
get_current_weather(location="Madrid", unit="celsius")

'{"location": "Madrid", "condition": "Sunny", "temperature": 18.0, "unit": "celsius"}'

In [3]:
import groq, httpx
print(groq.__version__)
print(httpx.__version__)


1.1.1
0.27.2


In [4]:
import os
import re
from dotenv import load_dotenv
from groq import Groq


# Remember to load the environment variables. You should have the Groq API Key in there :)
load_dotenv()

MODEL = "openai/gpt-oss-120b"
GROQ_CLIENT = Groq()

# Define the System Prompt as a constant
TOOL_SYSTEM_PROMPT = """
You are a function calling AI model. You are provided with function signatures within <tools></tools> XML tags. 
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug 
into functions. If the user does not specify a temperature unit, do not invent one; omit the unit argument so the tool can return both Celsius and Fahrenheit. Pay special attention to the properties 'types'. You should use those types as in a Python dict.
For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>}
</tool_call>

Here are the available tools:

<tools> {
    "name": "get_current_weather",
    "description": "Get the current weather in a given location. location (str): The city and state, e.g. Madrid, Barcelona. unit (str, optional): 'celsius' or 'fahrenheit'. If omitted because the user did not specify a unit, the tool returns both Celsius and Fahrenheit.",
    "parameters": {
        "properties": {
            "location": {
                "type": "str"
            },
            "unit": {
                "type": "str"
            }
        }
    }
}
</tools>
"""

In [5]:
tool_chat_history = [
    {
        "role": "system",
        "content": TOOL_SYSTEM_PROMPT
    }
]
agent_chat_history = []

user_msg = {
    "role": "user",
    "content": "What's the current temperature in Madrid, in Celsius?"
}

tool_chat_history.append(user_msg)
agent_chat_history.append(user_msg)

output = GROQ_CLIENT.chat.completions.create(
    messages=tool_chat_history,
    model=MODEL
).choices[0].message.content

print(output)

In [6]:
def parse_tool_call_str(tool_call_str: str):
    pattern = r'</?tool_call>'
    clean_tags = re.sub(pattern, '', tool_call_str)
    
    try:
        tool_call_json = json.loads(clean_tags)
        return tool_call_json
    except json.JSONDecodeError:
        return clean_tags
    except Exception as e:
        print(f"Unexpected error: {e}")
        return "There was some error parsing the Tool's output"

In [7]:
parsed_output = parse_tool_call_str(output)
parsed_output

''

In [8]:
result = get_current_weather(**parsed_output["arguments"])

TypeError: string indices must be integers, not 'str'

In [9]:
result

NameError: name 'result' is not defined

In [9]:
agent_chat_history.append({
    "role": "user",
    "content": f"Observation: {result}"
})

In [10]:
GROQ_CLIENT.chat.completions.create(
    messages=agent_chat_history,
    model=MODEL
).choices[0].message.content

'The current temperature in Madrid is **about\u202f18.3\u202f°C**.'

In [12]:
# Standard library imports used for data handling and path setup.
import json
import sys
from pathlib import Path

# Third-party package used for HTTP requests.
import requests

# Add the project root to Python path so local package imports work inside notebooks.
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import the local tool pattern module after path setup.
from src.tool_pattern import ToolAgent, tool


def fetch_top_hacker_news_stories(top_n: int):
    """
    Fetch the top stories from Hacker News.

    This function retrieves the top `top_n` stories from Hacker News using the Hacker News API.
    Each story contains the title, URL, score, author, and time of submission. The data is fetched
    from the official Firebase Hacker News API, which returns story details in JSON format.

    Args:
        top_n (int): The number of top stories to retrieve.
    """
    top_stories_url = "https://hacker-news.firebaseio.com/v0/topstories.json"

    try:
        response = requests.get(top_stories_url)
        response.raise_for_status()  # Check for HTTP errors

        # Get the top story IDs.
        top_story_ids = response.json()[:top_n]

        top_stories = []

        # For each story ID, fetch the story details.
        for story_id in top_story_ids:
            story_url = f"https://hacker-news.firebaseio.com/v0/item/{story_id}.json"
            story_response = requests.get(story_url)
            story_response.raise_for_status()  # Check for HTTP errors
            story_data = story_response.json()

            # Append the story title and URL (or other relevant info) to the list.
            top_stories.append(
                {
                    "title": story_data.get("title", "No title"),
                    "url": story_data.get("url", "No URL available"),
                }
            )

        return json.dumps(top_stories)

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        return []

In [13]:
json.loads(fetch_top_hacker_news_stories(top_n=5))

[{'title': 'LiteLLM Python package compromised by supply-chain attack',
  'url': 'https://github.com/BerriAI/litellm/issues/24512'},
 {'title': 'The bridge to wealth is being pulled up with AI',
  'url': 'https://danielhomola.com/m%20&%20e/ai/your-bridge-to-wealth-is-being-pulled-up/'},
 {'title': 'Microsoft\'s "Fix" for Windows 11: Flowers After the Beating',
  'url': 'https://www.sambent.com/microsofts-plan-to-fix-windows-11-is-gaslighting/'},
 {'title': 'Nanobrew: The fastest macOS package manager compatible with brew',
  'url': 'https://nanobrew.trilok.ai/'},
 {'title': 'Debunking Zswap and Zram Myths',
  'url': 'https://chrisdown.name/2026/03/24/zswap-vs-zram-when-to-use-what.html'}]

In [14]:
hn_tool = tool(fetch_top_hacker_news_stories)

In [15]:
hn_tool.name

'fetch_top_hacker_news_stories'

In [16]:
json.loads(hn_tool.fn_signature)

{'name': 'fetch_top_hacker_news_stories',
 'description': '\n    Fetch the top stories from Hacker News.\n\n    This function retrieves the top `top_n` stories from Hacker News using the Hacker News API. \n    Each story contains the title, URL, score, author, and time of submission. The data is fetched \n    from the official Firebase Hacker News API, which returns story details in JSON format.\n\n    Args:\n        top_n (int): The number of top stories to retrieve.\n    ',
 'parameters': {'properties': {'top_n': {'type': 'int'}}}}

In [9]:
tool_agent = ToolAgent(tools=[hn_tool])

NameError: name 'ToolAgent' is not defined

In [18]:
output = tool_agent.run(user_msg="Tell me your name")

BadRequestError: Error code: 400 - {'error': {'message': 'The model `llama3-groq-70b-8192-tool-use-preview` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}

In [ ]:
print(output)

In [ ]:
output = tool_agent.run(user_msg="Tell me the top 5 Hacker News stories right now")

In [ ]:
print(output)